# 测试模型 —— PINet 复振幅重建

本 notebook 用于加载训练好的 PINet 模型（v6 / v7 / CICDNet），
在**真实实验数据**上测试其复振幅重建效果（振幅 + 相位）。

### 数据格式（统一）
所有实验数据已整理为 `{name}_new.mat`，内部键名统一：
- `Shift_Samples`: `[H, W, N]` — N 张不同距离的衍射图
- `dist`: `[1, N]` — 对应的传播距离
- `label`: `[H, W]` — 复振幅真值（部分数据集有，部分没有）

### 使用方式
修改全局配置中的 `data_name`、`shift_sample_idx`、`distance_idx` 即可切换数据集和样本。

### 主要流程
1. 环境设置与导入
2. 全局配置
3. 数据加载与预处理（衍射图 + TF）
4. 标签可视化（如有 label）
5. 模型推理与可视化
6. 保存结果

In [ ]:
import sys
sys.path.insert(0, '/root/autodl-tmp/PINet_cpx_cl')

## 1. 环境设置与导入

In [ ]:
import os
import numpy as np
from scipy.io import loadmat

import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from src.model import PINet_cpx_v6, PINet_cpx_v7, PINet_cpx_CICDNet
from src.config import *
from src.utilities import (
    normalize_tensor,
    compute_psnr_skimage_single,
    compute_ssim_skimage_single,
)

## 2. 全局配置

设备、数据集选择、模型参数、可视化参数等全部集中在此 cell。
切换数据集只需修改 `data_name`。

In [ ]:
# ==================== 设备 ====================
device = DEVICE
print(f'Using device: {device}')

# ==================== 数据集选择 ====================
data_name = 'JZX1_type1'          # 完整文件名: {data_name}_new.mat
shift_sample_idx = 0             # 衍射图索引 (0 ~ N-1)
distance_idx = shift_sample_idx  # 距离索引 (通常与 shift_sample_idx 一致)

data_path = os.path.join(REAL_DATA_DIR, f'{data_name}_new.mat')

# ==================== 自动识别数据类型 (type1/type2) ====================
data_lower = data_name.lower()
if 'type1' in data_lower or data_name.upper() in ['HSTA', 'JZX1', 'MXWKY1']:
    data_type = 'type1'
    dx = 1.67e-6
elif 'type2' in data_lower or data_name.upper() in ['BEEWINGS', 'WKY']:
    data_type = 'type2'
    dx = 1.34e-6
else:
    data_type = 'type1'
    dx = PIXEL_SIZE
    print(f'  -> 无法从名称推断类型，使用 config 默认 {data_type}, dx = {dx:.2e} m')

print(f'  -> 识别为 {data_type}, dx = {dx:.2e} m')

# ==================== 模型配置 ====================
model_version = 'v6'           # 模型版本: 'v6' / 'v7' / 'CICDNet'
fold_iters = 6

checkpoint_epochs = list(range(0, 210, 50))

model_folder = os.path.join(SAVE_DIR, 'model_saved_pinet_compared4')


model_pattern = f'model_pinet_size256_epoch_{{epoch}}_batchsize4_1.5mm_to_3mm.pth'

# ==================== 图像预处理 ====================
transform = TRANSFORM

# ==================== 可视化参数 ====================
percent = 0.5

# ==================== 打印当前配置 ====================
print(f'\nDataset: {data_name}')
print(f'  File:          {data_path}')
print(f'  Type:          {data_type} (dx={dx*1e6:.2f} um)')
print(f'  Sample idx:    {shift_sample_idx}')
print(f'  Distance idx:  {distance_idx}')
print(f'Model: {model_version}, fold_iters={fold_iters}')
print(f'Epochs to test: {checkpoint_epochs}')
print(f'Model folder:   {model_folder}')

In [ ]:
# 物理参数（根据 data_type 自动选择）
print(f'Wavelength: {WAVELENGTH*1e9:.0f} nm, Pixel size: {dx*1e6:.2f} um ({data_type})')
print(f'Max spatial frequency (Nyquist): {1/(2*dx)*1e-3:.1f} lp/mm')
print('TF_torch will be recomputed from loaded data in Section 3.2')

## 3. 数据加载与预处理

从统一格式的 `{data_name}_new.mat` 加载数据，提取指定索引的衍射图和距离，计算传递函数 TF。

### 3.1 加载数据

所有数据集使用统一的键名：`Shift_Samples` / `dist` / `label`(可选)。

In [ ]:
# ==================== 加载 .mat 文件 ====================
data = loadmat(data_path)
print(f'Loaded: {data_path}')
print(f'Keys: {[k for k in data.keys() if not k.startswith("__")]}')

# ==================== 提取标准化数据 ====================
# 统一键名: Shift_Samples [H,W,N], dist [1,N], label [H,W] (可选)
all_Shift_Samples = data['Shift_Samples'].astype(np.float64)
dist_array = data['dist']
has_label = 'label' in data

print(f'Shift_Samples shape: {all_Shift_Samples.shape}, dtype: {all_Shift_Samples.dtype}')
print(f'dist shape: {dist_array.shape}, values: {dist_array[0]}')
print(f'Has label: {has_label}')

# 按索引提取当前样本
diffraction = all_Shift_Samples[:, :, shift_sample_idx]
dist_val = dist_array[0, distance_idx]
print(f'\nSelected: Shift_Samples[:,:,{shift_sample_idx}], dist[0,{distance_idx}] = {dist_val:.4f}')

# ==================== 标签处理与可视化 ====================
if has_label:
    label = data['label']
    print(f'label shape: {label.shape}')
    # 如果标签尺寸与衍射图不一致，裁剪对齐
    Hd, Wd = diffraction.shape[:2]
    if label.shape[:2] != (Hd, Wd):
        label = label[:Hd, :Wd]
        print(f'label cropped to: {label.shape}')

    label_amp = np.abs(label)
    label_phs = np.angle(label)

    vmin_label_amp = np.percentile(label_amp, percent)
    vmax_label_amp = np.percentile(label_amp, 100 - percent)
    vmin_label_phs = np.percentile(label_phs, percent)
    vmax_label_phs = np.percentile(label_phs, 100 - percent)
    print(f'Label amplitude range: [{vmin_label_amp:.3f}, {vmax_label_amp:.3f}]')
    print(f'Label phase range:      [{vmin_label_phs:.3f}, {vmax_label_phs:.3f}]')

    # 归一化显示
    label_amp_display = np.clip(label_amp, vmin_label_amp, vmax_label_amp)
    label_phs_display = np.clip(label_phs, vmin_label_phs, vmax_label_phs)
    label_amp_display = (label_amp_display - label_amp_display.min()) / (label_amp_display.max() - label_amp_display.min())
    label_phs_display = (label_phs_display - label_phs_display.min()) / (label_phs_display.max() - label_phs_display.min())

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(label_amp_display, cmap='gray')
    axes[0].set_title(f'Label Amplitude ({data_name})')
    axes[0].axis('off')
    axes[1].imshow(label_phs_display, cmap='gray')
    axes[1].set_title(f'Label Phase ({data_name})')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(label_amp.ravel(), bins=100)
    axes[0].set_title('Histogram of Label Amplitude')
    axes[0].set_xlabel('Amplitude')
    axes[0].set_ylabel('Count')
    axes[1].hist(label_phs.ravel(), bins=100)
    axes[1].set_title('Histogram of Label Phase')
    axes[1].set_xlabel('Phase (rad)')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    label = None
    print('No label in this dataset — skipping label visualization.')


### 3.2 衍射图预处理与传递函数 (TF)

对衍射图开根号增强对比度，并根据实际距离计算传递函数。

In [ ]:
# ==================== 衍射图预处理 ====================
H, W = diffraction.shape
Nx, Ny = H // 8 * 8, W // 8 * 8  # 对齐到 8 的倍数

diff_np = np.sqrt(diffraction)    # 开根号增强对比度
diff_np = diff_np[:Nx, :Ny]

plt.imshow(diff_np, cmap='gray')
plt.title(f'Diffraction Pattern ({data_name}, idx={shift_sample_idx})')
plt.axis('off')
plt.show()

diff_tensor = torch.from_numpy(diff_np).to(device).float()
print(f'Input tensor: shape={diff_tensor.shape}, range=[{diff_tensor.min():.3f}, {diff_tensor.max():.3f}]')

# ==================== 计算传递函数 (TF) ====================
z = dist_val * 1e-3              # 传播距离 (mm -> m)
k = 2 * np.pi / WAVELENGTH       # 波数
print(f'Using: dx = {dx:.2e} m ({data_type}), z = {z*1e3:.3f} mm')

Lx = Nx * dx
Ly = Ny * dx
fx = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Lx, Nx)
fy = np.linspace(-1 / (2 * dx), 1 / (2 * dx) - 1 / Ly, Ny)
FX, FY = np.meshgrid(fy, fx)
TF = np.exp(1j * z * np.sqrt(k ** 2 - (2 * np.pi * FX) ** 2 - (2 * np.pi * FY) ** 2))
TF_torch = torch.from_numpy(TF).to(device).to(torch.complex64)

print(f'TF_torch: shape={TF_torch.shape}, dtype={TF_torch.dtype}')
print(f'z = {z * 1e3:.3f} mm | dx = {dx * 1e6:.2f} um | Nx,Ny = {Nx},{Ny}')
print(f'Field of view: {Lx*1e3:.2f} mm x {Ly*1e3:.2f} mm')
print(f'Nyquist freq: {1/(2*dx)*1e-3:.1f} lp/mm')

## 4. 模型推理

加载不同 epoch 的模型 checkpoint，对比训练过程中的重建效果。

In [ ]:
def load_checkpoint(checkpoint_path, model):
    """加载模型 checkpoint。"""
    if os.path.isfile(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f'  -> Loaded: {os.path.basename(checkpoint_path)}')
    else:
        print(f'  -> NOT FOUND: {checkpoint_path}')


# 根据 model_version 选择模型类
MODEL_CLASSES = {
    'v6':      PINet_cpx_v6,
    'v7':      PINet_cpx_v7,
    'CICDNet': PINet_cpx_CICDNet,
}
ModelClass = MODEL_CLASSES[model_version]

print(f'Model pattern for {data_type}: {model_pattern.format(epoch="<epoch>")}')
print(f'Model folder: {model_folder}\n')

with torch.no_grad():
    for epoch in checkpoint_epochs:
        # 初始化模型
        model = ModelClass(fold_iters=fold_iters).to(device)
        model_path = model_pattern.format(epoch=epoch)
        checkpoint_path = os.path.join(model_folder, model_path)
        load_checkpoint(checkpoint_path, model)

        # 推理
        pred, y_rec = model(diff_tensor.unsqueeze(0).unsqueeze(0), TF_torch)
        pred_np = pred.squeeze(0).squeeze(0).detach().cpu().numpy()
        pred_amp_np = np.abs(pred_np)
        pred_phs_np = np.angle(pred_np)

        # 百分位裁剪并归一化显示
        vmin_amp = np.percentile(pred_amp_np, percent)
        vmax_amp = np.percentile(pred_amp_np, 100 - percent)
        vmin_phs = np.percentile(pred_phs_np, percent)
        vmax_phs = np.percentile(pred_phs_np, 100 - percent)

        pred_amp_display = np.clip(pred_amp_np, vmin_amp, vmax_amp)
        pred_phs_display = np.clip(pred_phs_np, vmin_phs, vmax_phs)
        pred_amp_display = (pred_amp_display - pred_amp_display.min()) / (pred_amp_display.max() - pred_amp_display.min())
        pred_phs_display = (pred_phs_display - pred_phs_display.min()) / (pred_phs_display.max() - pred_phs_display.min())

        print(f'Epoch {epoch:2d}: amp [{vmin_amp:.3f}, {vmax_amp:.3f}], phs [{vmin_phs:.3f}, {vmax_phs:.3f}]')

        # 可视化
        fig, axes = plt.subplots(2, 2, figsize=(15, 8), gridspec_kw={'height_ratios': [3, 1]})
        axes[0, 0].imshow(pred_amp_display, cmap='gray')
        axes[0, 0].set_title(f'Predicted Amplitude ({data_type}, {epoch} epochs)')
        axes[0, 0].axis('off')
        axes[0, 1].imshow(pred_phs_display, cmap='gray')
        axes[0, 1].set_title(f'Predicted Phase ({data_type}, {epoch} epochs)')
        axes[0, 1].axis('off')
        axes[1, 0].hist(pred_amp_np.ravel(), bins=100)
        axes[1, 0].set_title('Histogram of Amplitude')
        axes[1, 0].set_xlabel('Amplitude')
        axes[1, 0].set_ylabel('Count')
        axes[1, 1].hist(pred_phs_np.ravel(), bins=100)
        axes[1, 1].set_title('Histogram of Phase')
        axes[1, 1].set_xlabel('Phase (rad)')
        axes[1, 1].set_ylabel('Count')
        plt.tight_layout()
        plt.show()

## 5. 保存结果

In [ ]:
# 保存最后一个 epoch 的预测结果
save_path_amp = os.path.join(model_folder, f'amp_pred_{data_name}_{model_version}_fold_iters_{fold_iters}.png')
save_path_phs = os.path.join(model_folder, f'phs_pred_{data_name}_{model_version}_fold_iters_{fold_iters}.png')

plt.imsave(save_path_amp, pred_amp_display, cmap='gray')
plt.imsave(save_path_phs, pred_phs_display, cmap='gray')
print(f'Amplitude saved to: {save_path_amp}')
print(f'Phase saved to:    {save_path_phs}')